# Perturb-Seqr Similar & Opposite Perturbations Appyter

This appyter takes a query signature — an **up-regulated** gene list and a **down-regulated** gene list from any experiment, drug treatment, or gene perturbation — and searches [Perturb-Seqr](https://perturbseqr.maayanlab.cloud), a connectivity-mapping database integrating 9 small-molecule and 7 single-gene perturbation signature collections, to find:

- **Mimickers** — drugs or genes whose signatures move genes in the *same* direction as your input (similar biological effect).
- **Reversers** — drugs or genes whose signatures move genes in the *opposite* direction (potential reversal/repurposing candidates).

Results are reported separately for **drug** and **gene (knockout/knockdown)** perturbations, in each of the two directions, giving four ranked tables and bar charts, plus a combined interactive overview plot. You can also drill down into which specific genes drove the single strongest mimicker and reverser hit.

For simplicity, the primary inputs are your up/down gene lists and a handful of display parameters. Other parameters are set to reasonable defaults — download the notebook and rerun with different settings if you wish.

In [ ]:
#%%appyter init
from appyter import magic
magic.init(lambda _=globals: _())

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
output_notebook()

In [ ]:
%%appyter hide_code

{% do SectionField(
    name='section1',
    title='1. Submit Your Up/Down Gene Sets',
    subtitle='Provide the genes that go up and the genes that go down under your condition of interest (a drug treatment, a gene knockdown/overexpression, a disease state, etc.). Paste lists directly or upload text files (one gene per row). Default example gene lists are provided.',
) %}
{% do SectionField(
    name='section2',
    title='2. Display Parameters',
    subtitle='Set a query name and choose how many top hits to show in each results table/chart.',
) %}

In [ ]:
%%appyter hide_code

{% set up_kind = TabField(
    name='up_kind',
    label='Up-Regulated Gene List',
    default='Paste',
    description='Paste or upload the genes that go UP under your condition of interest',
    required=True,
    choices={
        'Paste': [
            TextListField(
                name='up_input',
                label='Up-Regulated Genes',
                description='Paste your up-regulated gene list (one gene per row).',
                default=[
                    'CDKN1A', 'MDM2', 'BAX', 'GADD45A', 'BBC3', 'SESN1', 'RRM2B', 'TP53I3'
                ],
                section='section1'
            ),
        ],
        'Upload': [
            FileField(
                name='up_filename',
                label='Up-Regulated Gene List File',
                default='',
                description='Upload the up-regulated gene list as a text file (one gene per row).',
                section='section1'
            ),
        ],
    },
    section='section1',
) %}

{% set down_kind = TabField(
    name='down_kind',
    label='Down-Regulated Gene List',
    default='Paste',
    description='Paste or upload the genes that go DOWN under your condition of interest',
    required=True,
    choices={
        'Paste': [
            TextListField(
                name='down_input',
                label='Down-Regulated Genes',
                description='Paste your down-regulated gene list (one gene per row).',
                default=[
                    'MYC', 'CCNB1', 'CCNE1', 'CDK1', 'BIRC5', 'TOP2A', 'MKI67', 'PLK1'
                ],
                section='section1'
            ),
        ],
        'Upload': [
            FileField(
                name='down_filename',
                label='Down-Regulated Gene List File',
                default='',
                description='Upload the down-regulated gene list as a text file (one gene per row).',
                section='section1'
            ),
        ],
    },
    section='section1',
) %}

{% set query_name = StringField(
    name='query_name',
    label='Query Name',
    description='A label for this analysis, used in output titles and the saved report.',
    default='my_signature',
    section='section2'
) %}

{% set top_n_drugs = IntField(
    name='top_n_drugs',
    label='Top Drugs to Display (per direction)',
    description='Number of top drug perturbations to show in each of the Mimicker Drugs and Reverser Drugs tables/charts.',
    default=12,
    min=1,
    max=50,
    section='section2'
) %}

{% set top_n_genes = IntField(
    name='top_n_genes',
    label='Top Gene Perturbations to Display (per direction)',
    description='Number of top gene (knockout/knockdown) perturbations to show in each of the Mimicker Genes and Reverser Genes tables/charts.',
    default=12,
    min=1,
    max=50,
    section='section2'
) %}

In [ ]:
%%appyter code_exec

{%- if up_kind.raw_value == 'Paste' %}
up_input = {{ up_kind.value[0] }}
{%- else %}
up_filename = {{ up_kind.value[0] }}
{%- endif %}
{%- if down_kind.raw_value == 'Paste' %}
down_input = {{ down_kind.value[0] }}
{%- else %}
down_filename = {{ down_kind.value[0] }}
{%- endif %}
query_name = {{ query_name }}
top_n_drugs = {{ top_n_drugs }}
top_n_genes = {{ top_n_genes }}

In [ ]:
%%appyter code_exec

{%- if up_kind.raw_value == 'Paste' %}
genes_up = [x.strip() for x in up_input]
{%- else %}
_f = open(up_filename, 'r')
genes_up = [x.strip() for x in _f.readlines()]
_f.close()
{%- endif %}
{%- if down_kind.raw_value == 'Paste' %}
genes_down = [x.strip() for x in down_input]
{%- else %}
_f = open(down_filename, 'r')
genes_down = [x.strip() for x in _f.readlines()]
_f.close()
{%- endif %}
genes_up = [g for g in genes_up if g]
genes_down = [g for g in genes_down if g]

# Error handling
class NoResults(Exception):
    pass

class APIFailure(Exception):
    pass

print(f'Up-regulated gene set loaded: {len(genes_up)} genes')
print(f'Down-regulated gene set loaded: {len(genes_down)} genes')

In [ ]:
# Shared helper: horizontal bar chart used to visualize each category's top results
def make_bar_chart(labels, scores, title, xlabel, colors='lightskyblue', legend_handles=None, figsize=None):
    labels = list(labels)
    scores = list(scores)
    if figsize is None:
        figsize = (9, max(3, 0.4 * len(labels)))
    order = np.argsort(scores)  # ascending so the highest score plots at the top
    labels_sorted = [labels[i] for i in order]
    scores_sorted = [scores[i] for i in order]
    if isinstance(colors, (list, tuple, np.ndarray)):
        colors_sorted = [colors[i] for i in order]
    else:
        colors_sorted = colors

    plt.figure(figsize=figsize)
    ax = plt.gca()
    ax.barh(labels_sorted, scores_sorted, color=colors_sorted, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    if legend_handles:
        ax.legend(handles=legend_handles, loc='lower right', frameon=False)
    plt.tight_layout()
    return ax

## Step 1: Validate Genes Against the Perturb-Seqr Background
Genes not recognized in the Perturb-Seqr background are flagged and dropped so the enrichment isn't silently run on an incomplete set.

In [ ]:
PERTURBSEQR_URL = 'https://perturbseqr.maayanlab.cloud/graphql'

def get_perturbseqr_valid_genes(genes):
    query = {
        "query": """query GenesQuery($genes: [String]!) {
            geneMap2(genes: $genes) {
                nodes {
                    gene
                    geneInfo {
                        symbol
                        }
                    }
                }
            }""",
        "variables": {"genes": genes},
        "operationName": "GenesQuery"
    }
    r = requests.post(PERTURBSEQR_URL, json=query)
    if not r.ok:
        raise APIFailure
    res = r.json()
    return [g['geneInfo']['symbol'] for g in res['data']['geneMap2']['nodes'] if g['geneInfo'] is not None]

try:
    valid_up = get_perturbseqr_valid_genes(genes_up)
    valid_down = get_perturbseqr_valid_genes(genes_down)

    print(f"Valid up genes ({len(valid_up)}/{len(genes_up)}):   {valid_up}")
    print(f"Valid down genes ({len(valid_down)}/{len(genes_down)}): {valid_down}")

    missing_up = set(genes_up) - set(valid_up)
    missing_down = set(genes_down) - set(valid_down)
    if missing_up or missing_down:
        print(f"\nNot found in background — up: {missing_up}, down: {missing_down}")
except APIFailure:
    valid_up, valid_down = genes_up, genes_down
    display(HTML("<div style='font-size:1.05rem; padding:0.5rem 0;'><b>Gene validation call failed — proceeding with the original input lists.</b></div>"))

## Step 2: Perturb-Seqr — Similar (Mimicker) & Opposite (Reverser) Perturbations
The validated up/down gene sets are queried against Perturb-Seqr's paired enrichment endpoint, which runs a Fisher's exact test comparing your signature against every drug and gene perturbation gene set in the database:

- **`pvalueMimic` / `oddsRatioMimic`** — strength of same-direction overlap (mimicker — **most similar**).
- **`pvalueReverse` / `oddsRatioReverse`** — strength of opposite-direction overlap (reverser — **most opposite**).

Results are split into four separate tables and charts so drug and gene perturbations, and their mimicker/reverser relationships, can each be inspected on their own terms:

- **Mimicker Drugs** — small-molecule perturbations whose signature moves genes in the *same* direction as your input.
- **Reverser Drugs** — small-molecule perturbations whose signature moves genes in the *opposite* direction (potential repurposing candidates).
- **Mimicker Genes** — gene knockout/knockdown perturbations whose signature moves genes in the *same* direction as your input.
- **Reverser Genes** — gene knockout/knockdown perturbations whose signature moves genes in the *opposite* direction.

Drug vs. gene perturbations are distinguished by querying Perturb-Seqr's knockout/knockdown-only filter, the same convention used elsewhere in the Perturb-Seqr Appyter family.

In [ ]:
def query_perturbseqr_paired(genes_up, genes_down, first=500, filter_ko=False):
    query = {
        "operationName": "PairEnrichmentQuery",
        "variables": {
            "filterTerm": "",
            "offset": 0,
            "first": first,
            "filterFda": False,
            "sortBy": "pvalue_mimic",
            "filterKo": filter_ko,
            "topN": 1000,
            "pvalueLe": 0.05,
            "genesUp": genes_up,
            "genesDown": genes_down
        },
        "query": """query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!, $filterTerm: String = "", $offset: Int = 0, $first: Int = 10, $filterFda: Boolean = false, $sortBy: String = "", $filterKo: Boolean = false, $topN: Int = 10000, $pvalueLe: Float = 0.05) {
          currentBackground {
            pairedEnrich(
              filterTerm: $filterTerm
              offset: $offset
              first: $first
              filterFda: $filterFda
              sortby: $sortBy
              filterKo: $filterKo
              topN: $topN
              pvalueLe: $pvalueLe
              genesDown: $genesDown
              genesUp: $genesUp
              ) {
                totalCount
                consensusCount
                consensus {
                  drug
                  oddsRatio
                  pvalue
                  adjPvalue
                  approved
                  countSignificant
                  countInsignificant
                  countUpSignificant
                  pvalueUp
                  adjPvalueUp
                  oddsRatioUp
                  pvalueDown
                  adjPvalueDown
                  oddsRatioDown
                  }
                  nodes {
                    adjPvalueMimic
                    adjPvalueReverse
                    mimickerOverlap
                    oddsRatioMimic
                    oddsRatioReverse
                    pvalueMimic
                    pvalueReverse
                    reverserOverlap
                    geneSet {
                      nodes {
                        id
                        nGeneIds
                        term
                        geneSetFdaCountsById {
                          nodes {
                            count
                            approved
                            }
                          }
                        }
                      }
                    }
                  }
                }
              }
        """
    }
    r = requests.post(PERTURBSEQR_URL, json=query)
    if not r.ok:
        raise APIFailure
    res = r.json()
    consensus = (res.get('data') or {}).get('currentBackground', {}).get('pairedEnrich', {}).get('consensus', [])
    nodes = (res.get('data') or {}).get('currentBackground', {}).get('pairedEnrich', {}).get('nodes', [])
    if not consensus:
        raise NoResults

    df_consensus = pd.DataFrame(consensus).rename(columns={
        'drug': 'perturbation',
        'pvalueUp': 'pvalueMimic', 'adjPvalueUp': 'adjPvalueMimic', 'oddsRatioUp': 'oddsRatioMimic',
        'pvalueDown': 'pvalueReverse', 'adjPvalueDown': 'adjPvalueReverse', 'oddsRatioDown': 'oddsRatioReverse',
    })

    df_nodes = pd.DataFrame(nodes)
    if not df_nodes.empty:
        df_nodes['term'] = df_nodes['geneSet'].map(lambda t: t['nodes'][0]['term'].split(' ')[0])
        df_nodes['geneSetIdUp'] = df_nodes['geneSet'].map(
            lambda t: next((n['id'] for n in t['nodes'] if ' up' in n['term']), None)
        )
        df_nodes['geneSetIdDown'] = df_nodes['geneSet'].map(
            lambda t: next((n['id'] for n in t['nodes'] if ' down' in n['term']), None)
        )
        df_nodes = df_nodes.drop(columns=['geneSet'])

    return df_consensus, df_nodes


def build_direction_table(df, top_n, direction):
    """Rank a consensus dataframe by the mimic- or reverse-direction p-value and
    build a display-ready table for that single direction."""
    pcol, ocol = ('pvalueMimic', 'oddsRatioMimic') if direction == 'mimic' else ('pvalueReverse', 'oddsRatioReverse')
    d = df.dropna(subset=[pcol]).copy()
    d = d.sort_values(pcol, ascending=True).head(top_n).reset_index(drop=True)
    d['neg_log_pvalue'] = -np.log10(d[pcol].clip(lower=1e-300))
    table = pd.DataFrame({
        'Rank': range(1, len(d) + 1),
        'Perturbation': d['perturbation'],
        'Odds Ratio': d[ocol].round(2),
        'p-value': d[pcol].map(lambda x: f'{x:.2e}'),
        'FDA Approved': d['approved'],
    })
    return d, table


def show_perturbation_section(d, table, title, xlabel, color, file_stub, fig_caption):
    """Display a table + bar chart for one of the four (mimic/reverse x drug/gene)
    categories. Returns the saved chart filename, or None if there were no
    results for this category."""
    if d.empty:
        display(HTML("<div style='font-size:1.05rem; padding:0.5rem 0;'><b>No results in this category for the current gene set.</b></div>"))
        return None
    display(HTML(f'<strong>{title}</strong>'))
    display(HTML(table.to_html(index=False)))
    make_bar_chart(
        table['Perturbation'], d['neg_log_pvalue'],
        title=title,
        xlabel=xlabel,
        colors=color
    )
    bar_file = f"{file_stub}.png".replace(' ', '_')
    plt.savefig(bar_file, bbox_inches='tight')
    plt.show()
    display(Markdown(fig_caption))
    return bar_file

In [ ]:
mimic_drugs_df = reverse_drugs_df = mimic_genes_df = reverse_genes_df = None
mimic_drugs_table = reverse_drugs_table = mimic_genes_table = reverse_genes_table = None
mimic_drugs_bar_file = reverse_drugs_bar_file = mimic_genes_bar_file = reverse_genes_bar_file = None
perturbseqr_link = 'https://perturbseqr.maayanlab.cloud'
perturbseqr_ok = False
nodes_all = pd.DataFrame()

try:
    # All perturbations (drugs + genes together)
    consensus_all, nodes_all = query_perturbseqr_paired(valid_up, valid_down, first=500, filter_ko=False)
    # Knockout/knockdown-only perturbations, used to identify which rows above are genes
    try:
        consensus_ko, _ = query_perturbseqr_paired(valid_up, valid_down, first=500, filter_ko=True)
    except NoResults:
        consensus_ko = pd.DataFrame(columns=consensus_all.columns)

    gene_names = set(consensus_ko['perturbation']) if not consensus_ko.empty else set()
    drugs_all = consensus_all[~consensus_all['perturbation'].isin(gene_names)].reset_index(drop=True)
    genes_all = consensus_ko.reset_index(drop=True)
    perturbseqr_ok = True

    # --- Mimicker Drugs ---
    mimic_drugs_df, mimic_drugs_table = build_direction_table(drugs_all, top_n_drugs, 'mimic')
    caption_mimic_drugs = f"**Table 1a. Top {top_n_drugs} drug perturbations for `{query_name}` that MIMIC (same direction as) the query signature.**"
    mimic_drugs_bar_file = show_perturbation_section(
        mimic_drugs_df, mimic_drugs_table,
        title=f'Top {top_n_drugs} Mimicker Drugs (Perturb-Seqr)',
        xlabel='-log10(p-value)', color='#B0B0B0',
        file_stub=f"{query_name}_mimic_drugs_bar",
        fig_caption=f"**Figure 1. Top {len(mimic_drugs_df)} most SIMILAR drug perturbations for `{query_name}`,** ranked by -log10(p-value) for same-direction overlap."
    )
    display(Markdown(caption_mimic_drugs))

    # --- Reverser Drugs ---
    reverse_drugs_df, reverse_drugs_table = build_direction_table(drugs_all, top_n_drugs, 'reverse')
    caption_reverse_drugs = f"**Table 1b. Top {top_n_drugs} drug perturbations for `{query_name}` that REVERSE (opposite direction from) the query signature — potential repurposing candidates.**"
    reverse_drugs_bar_file = show_perturbation_section(
        reverse_drugs_df, reverse_drugs_table,
        title=f'Top {top_n_drugs} Reverser Drugs (Perturb-Seqr)',
        xlabel='-log10(p-value)', color='#4C72B0',
        file_stub=f"{query_name}_reverse_drugs_bar",
        fig_caption=f"**Figure 2. Top {len(reverse_drugs_df)} most OPPOSITE drug perturbations for `{query_name}`,** ranked by -log10(p-value) for opposite-direction overlap."
    )
    display(Markdown(caption_reverse_drugs))

    # --- Mimicker Genes ---
    mimic_genes_df, mimic_genes_table = build_direction_table(genes_all, top_n_genes, 'mimic')
    caption_mimic_genes = f"**Table 1c. Top {top_n_genes} gene knockout/knockdown perturbations for `{query_name}` that MIMIC the query signature.**"
    mimic_genes_bar_file = show_perturbation_section(
        mimic_genes_df, mimic_genes_table,
        title=f'Top {top_n_genes} Mimicker Genes (Perturb-Seqr)',
        xlabel='-log10(p-value)', color='#ffa600',
        file_stub=f"{query_name}_mimic_genes_bar",
        fig_caption=f"**Figure 3. Top {len(mimic_genes_df)} most SIMILAR gene perturbations for `{query_name}`,** ranked by -log10(p-value) for same-direction overlap."
    )
    display(Markdown(caption_mimic_genes))

    # --- Reverser Genes ---
    reverse_genes_df, reverse_genes_table = build_direction_table(genes_all, top_n_genes, 'reverse')
    caption_reverse_genes = f"**Table 1d. Top {top_n_genes} gene knockout/knockdown perturbations for `{query_name}` that REVERSE the query signature.**"
    reverse_genes_bar_file = show_perturbation_section(
        reverse_genes_df, reverse_genes_table,
        title=f'Top {top_n_genes} Reverser Genes (Perturb-Seqr)',
        xlabel='-log10(p-value)', color='#7a5195',
        file_stub=f"{query_name}_reverse_genes_bar",
        fig_caption=f"**Figure 4. Top {len(reverse_genes_df)} most OPPOSITE gene perturbations for `{query_name}`,** ranked by -log10(p-value) for opposite-direction overlap."
    )
    display(Markdown(caption_reverse_genes))

    # --- Combined interactive overview across all four categories ---
    def _tag(d, cat):
        if d is None or d.empty:
            return pd.DataFrame()
        t = d.copy()
        t['category'] = cat
        t['x'] = t['oddsRatioMimic'] if cat.startswith('Mimic') else t['oddsRatioReverse']
        return t

    combined = pd.concat([
        _tag(mimic_drugs_df, 'Mimic Drug'),
        _tag(reverse_drugs_df, 'Reverse Drug'),
        _tag(mimic_genes_df, 'Mimic Gene'),
        _tag(reverse_genes_df, 'Reverse Gene'),
    ], ignore_index=True)

    if not combined.empty:
        cat_colors = {'Mimic Drug': '#B0B0B0', 'Reverse Drug': '#4C72B0', 'Mimic Gene': '#ffa600', 'Reverse Gene': '#7a5195'}
        combined['colors'] = combined['category'].map(cat_colors)

        source = ColumnDataSource(data=dict(
            x=combined['x'],
            y=combined['neg_log_pvalue'],
            perturbation=combined['perturbation'],
            category=combined['category'],
            colors=combined['colors'],
        ))
        hover = HoverTool(tooltips=[
            ('Perturbation', '@perturbation'),
            ('Category', '@category'),
            ('-log10(p-value)', '@y'),
            ('Odds Ratio', '@x'),
        ])
        overview_plot = figure(
            width=700, height=500,
            tools=[hover, 'pan', 'wheel_zoom', 'reset', 'save'],
            title='Perturb-Seqr Overview: Mimickers & Reversers (Drugs & Genes)'
        )
        overview_plot.circle('x', 'y', size=10, fill_color='colors', line_color='colors', fill_alpha=0.7, line_alpha=0.7, source=source)
        overview_plot.xaxis.axis_label = 'Odds Ratio'
        overview_plot.yaxis.axis_label = '-log10(p-value)'
        overview_plot.output_backend = 'svg'
        show(overview_plot)
        display(Markdown(f"**Figure 5. Combined overview of the top {len(combined)} perturbations shown in Tables 1a-1d for `{query_name}`,** colored by category (grey = mimicker drugs, blue = reverser drugs, orange = mimicker genes, purple = reverser genes). Hover over a point for details."))

except APIFailure:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>Unable to retrieve results because of a bad response from the Perturb-Seqr API</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No perturbations were returned for this gene set</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different gene list.</div>"))

## Step 3: Which Genes Drove the Top Hit?
Take the single strongest mimicker and reverser perturbation instance (across drugs and genes combined) and pull the exact genes from your input signature that overlap with that perturbation's gene set.

In [ ]:
def get_overlap(genes, gene_set_id):
    query = {
        "operationName": "OverlapQuery",
        "variables": {"id": gene_set_id, "genes": genes},
        "query": """query OverlapQuery($id: UUID!, $genes: [String]!) {geneSet(id: $id) {
        overlap(genes: $genes) {
          nodes {
            symbol
          }   }}}"""
    }
    r = requests.post(PERTURBSEQR_URL, json=query)
    if not r.ok:
        raise APIFailure
    res = r.json()
    return [item['symbol'] for item in res['data']['geneSet']['overlap']['nodes']]


def get_perturbseqr_up_dn_overlap(genes_up, genes_down, id_up, id_down, overlap_type):
    if overlap_type == 'mimicker':
        up_up = get_overlap(genes_up, id_up)
        dn_dn = get_overlap(genes_down, id_down)
        return sorted(set(up_up) | set(dn_dn))
    elif overlap_type == 'reverser':
        up_dn = get_overlap(genes_up, id_down)
        dn_up = get_overlap(genes_down, id_up)
        return sorted(set(up_dn) | set(dn_up))


if perturbseqr_ok and not nodes_all.empty:
    try:
        top_mimicker_row = nodes_all.sort_values('pvalueMimic').iloc[0]
        mimicker_overlap_genes = get_perturbseqr_up_dn_overlap(
            valid_up, valid_down,
            top_mimicker_row['geneSetIdUp'], top_mimicker_row['geneSetIdDown'],
            overlap_type='mimicker'
        )
        display(HTML(f"<div style='font-size:1.05rem;'><b>Top overall mimicker:</b> {top_mimicker_row['term']} "
                      f"(p-value = {top_mimicker_row['pvalueMimic']:.2e}, odds ratio = {top_mimicker_row['oddsRatioMimic']:.2f})</div>"))
        display(HTML(f"<div>Overlapping genes: {', '.join(mimicker_overlap_genes) if mimicker_overlap_genes else '(none found)'}</div>"))

        top_reverser_row = nodes_all.sort_values('pvalueReverse').iloc[0]
        reverser_overlap_genes = get_perturbseqr_up_dn_overlap(
            valid_up, valid_down,
            top_reverser_row['geneSetIdUp'], top_reverser_row['geneSetIdDown'],
            overlap_type='reverser'
        )
        display(HTML(f"<div style='font-size:1.05rem; padding-top:1rem;'><b>Top overall reverser:</b> {top_reverser_row['term']} "
                      f"(p-value = {top_reverser_row['pvalueReverse']:.2e}, odds ratio = {top_reverser_row['oddsRatioReverse']:.2f})</div>"))
        display(HTML(f"<div>Overlapping genes: {', '.join(reverser_overlap_genes) if reverser_overlap_genes else '(none found)'}</div>"))
    except APIFailure:
        display(HTML("<div style='font-size:1.05rem; padding:0.5rem 0;'><b>Unable to retrieve gene-level overlap details.</b></div>"))
else:
    display(HTML("<div style='font-size:1.05rem; padding:0.5rem 0;'><b>Skipping overlap drill-down — no Perturb-Seqr results available.</b></div>"))

## Link to Perturb-Seqr

In [ ]:
if perturbseqr_ok:
    display(HTML(f"<div style='font-size:1.25rem; padding:1rem 0;'><a href='{perturbseqr_link}' target='_blank'>Explore the complete Perturb-Seqr connectivity mapping database online.</a></div>"))
else:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No Perturb-Seqr results available for the current query</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different input list.</div>"))

## Save Full Report
Combine the results and charts from all four categories into a single standalone HTML report file that can be downloaded and shared.

In [ ]:
from datetime import datetime

def _safe_table_html(table_name, caption_name, image_file=None):
    df = globals().get(table_name)
    caption = globals().get(caption_name, '')
    if df is not None and len(df) > 0:
        img_html = f"<img src='{image_file}' style='max-width:100%; margin-top:1rem;'>" if image_file else ''
        return f"<div>{df.to_html(index=False)}</div>{img_html}<p>{caption}</p>"
    else:
        return "<p><i>No results were available for this category.</i></p>"

report_sections = []
report_sections.append("<h2>Perturb-Seqr &mdash; Similar &amp; Opposite Perturbation Candidates</h2>")
report_sections.append("<h3>Mimicker Drugs (Most Similar)</h3>")
report_sections.append(_safe_table_html('mimic_drugs_table', 'caption_mimic_drugs', image_file=globals().get('mimic_drugs_bar_file')))
report_sections.append("<h3>Reverser Drugs (Most Opposite)</h3>")
report_sections.append(_safe_table_html('reverse_drugs_table', 'caption_reverse_drugs', image_file=globals().get('reverse_drugs_bar_file')))
report_sections.append("<h3>Mimicker Genes (Most Similar)</h3>")
report_sections.append(_safe_table_html('mimic_genes_table', 'caption_mimic_genes', image_file=globals().get('mimic_genes_bar_file')))
report_sections.append("<h3>Reverser Genes (Most Opposite)</h3>")
report_sections.append(_safe_table_html('reverse_genes_table', 'caption_reverse_genes', image_file=globals().get('reverse_genes_bar_file')))

if globals().get('perturbseqr_ok'):
    report_sections.append(f"<p><a href='{perturbseqr_link}' target='_blank'>Explore the complete Perturb-Seqr connectivity mapping database online.</a> (The interactive overview plot for this run is available in the notebook.)</p>")

html_report = f"""<html>
<head>
<meta charset="utf-8">
<title>Perturb-Seqr Report - {query_name}</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 2rem; color: #222; }}
  table {{ border-collapse: collapse; margin-bottom: 1rem; }}
  th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
  th {{ background-color: #f2f2f2; }}
  h1 {{ border-bottom: 2px solid #333; padding-bottom: 0.5rem; }}
  h2 {{ margin-top: 2rem; color: #333; }}
  h3 {{ margin-top: 1.5rem; color: #555; }}
</style>
</head>
<body>
<h1>Perturb-Seqr Similar &amp; Opposite Perturbations: {query_name}</h1>
<p>Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
{''.join(report_sections)}
</body>
</html>
"""

report_filename = f"{query_name}_perturbseqr_report.html".replace(' ', '_')
with open(report_filename, 'w') as f:
    f.write(html_report)

display(HTML(f"<div style='font-size:1.25rem; padding:1rem 0;'>Full report saved to <code>{report_filename}</code></div>"))
display(HTML(f'<div>Download full report: <a href="{report_filename}" target=_blank>{report_filename}</a></div>'))